In [4]:
import gymnasium as gym
import numpy as np

def random_policy(state, env):
    """
    Defines the policy to evaluate.
    In this case, a completely random policy that selects actions uniformly.
    """
    return env.action_space.sample()

def monte_carlo_prediction(env, policy, num_episodes=5000, gamma=0.95):
    """
    Computes the state-value function V for a given policy using 
    First-Visit Monte Carlo prediction.
    """
    num_states = env.observation_space.n
    
    # Initialize trackers
    V = np.zeros(num_states)
    returns_sum = np.zeros(num_states)
    returns_count = np.zeros(num_states)

    print(f"Starting First-Visit Monte Carlo Prediction for {num_episodes} episodes...")

    for episode in range(1, num_episodes + 1):
        # 1. Generate an entire episode trajectory
        episode_history = []
        state, info = env.reset()
        done = False

        while not done:
            action = policy(state, env)
            next_state, reward, terminated, truncated, info = env.step(action)
            episode_history.append((state, reward))
            
            state = next_state
            done = terminated or truncated

        # 2. Process the accumulated rewards backwards (Backward Pass)
        G = 0
        visited_states = set()
        
        # Loop from the last time-step T-1 down to 0
        for t in range(len(episode_history) - 1, -1, -1):
            state, reward = episode_history[t]
            G = gamma * G + reward
            
            # First-visit MC check
            if state not in visited_states:
                visited_states.add(state)
                returns_sum[state] += G
                returns_count[state] += 1
                V[state] = returns_sum[state] / returns_count[state]

        # Log progress every 1,000 episodes
        if episode % 1000 == 0:
            print(f" -> Completed Episode {episode}/{num_episodes}...")

    print("Prediction complete!\n")
    return V

if __name__ == "__main__":
    # Initialize the Taxi-v4 environment
    env = gym.make("Taxi-v4")
    
    # Run the Monte Carlo Evaluation loop
    state_values = monte_carlo_prediction(env, random_policy, num_episodes=5000, gamma=0.95)
    env.close()

    # Display results
    print("Sample State Value Estimates:")
    print("-" * 40)
    
    # Safely find states using the returned state_values array
    visited_state_indices = np.where(state_values != 0)[0]
    
    if len(visited_state_indices) == 0:
        print("All estimated values are zero. The random policy did not hit meaningful rewards.")
    else:
        for state_id in visited_state_indices[:10]:
            print(f"State ID {state_id:03d} | Estimated Value V(s): {state_values[state_id]:.4f}")


Starting First-Visit Monte Carlo Prediction for 5000 episodes...
 -> Completed Episode 1000/5000...
 -> Completed Episode 2000/5000...
 -> Completed Episode 3000/5000...
 -> Completed Episode 4000/5000...
 -> Completed Episode 5000/5000...
Prediction complete!

Sample State Value Estimates:
----------------------------------------
State ID 001 | Estimated Value V(s): -57.2289
State ID 002 | Estimated Value V(s): -56.0740
State ID 003 | Estimated Value V(s): -60.5901
State ID 004 | Estimated Value V(s): -61.9019
State ID 006 | Estimated Value V(s): -58.1977
State ID 007 | Estimated Value V(s): -58.4599
State ID 008 | Estimated Value V(s): -64.3848
State ID 009 | Estimated Value V(s): -57.0136
State ID 011 | Estimated Value V(s): -56.6932
State ID 012 | Estimated Value V(s): -59.9379
